# Markov Decision Processes (MDP) — Q-Value Iteration

This notebook implements **Q-value iteration** for a small Markov Decision Process (MDP).

## Goal

We want to learn the best action to take in every state by calculating:

\[
Q^*(s,a)
\]

The Q-value tells us:

> **How good is it to take action `a` while being in state `s`, assuming the agent behaves optimally afterward?**

The update rule used in this notebook is:

\[
Q_{k+1}(s,a)=\sum_{s'}T(s,a,s')\left[R(s,a,s')+\gamma\max_{a'}Q_k(s',a')\right]
\]

Where:
- `T(s, a, s')` = transition probability
- `R(s, a, s')` = immediate reward
- `γ` = discount factor
- `max Q(s', a')` = best possible future Q-value


## 1. Import NumPy

NumPy is used to work with arrays and numerical values in the MDP.


In [ ]:
# Import NumPy and use the short name "np"
import numpy as np


## 2. Define the Markov Decision Process

We have:

- **3 states:** `s0`, `s1`, `s2`
- **3 possible actions:** `a0`, `a1`, `a2`

Not every action is available in every state.

### Transition probabilities

The structure is:

```text
transition_probabilities[current_state][action][next_state]
```

For example:

```text
transition_probabilities[2][1][0] = 0.8
```

means:

> From state `s2`, after taking action `a1`, there is an 80% probability of reaching state `s0`.

### Rewards

The structure is:

```text
rewards[current_state][action][next_state]
```

For example:

```text
rewards[2][1][0] = +40
```

means the agent receives a reward of **+40** when it moves from `s2` to `s0` after taking action `a1`.


In [ ]:
# Transition probabilities: T(s, a, s')
# Each inner list contains probabilities of reaching [s0, s1, s2].

transition_probabilities = [
    # State s0
    [
        [0.7, 0.3, 0.0],  # Action a0: 70% -> s0, 30% -> s1
        [1.0, 0.0, 0.0],  # Action a1: 100% -> s0
        [0.8, 0.2, 0.0],  # Action a2: 80% -> s0, 20% -> s1
    ],

    # State s1
    [
        [0.0, 1.0, 0.0],  # Action a0: 100% -> s1
        None,             # Action a1 is not possible in s1
        [0.0, 0.0, 1.0],  # Action a2: 100% -> s2
    ],

    # State s2
    [
        None,             # Action a0 is not possible in s2
        [0.8, 0.1, 0.1],  # Action a1: 80% -> s0, 10% -> s1, 10% -> s2
        None,             # Action a2 is not possible in s2
    ]
]


# Rewards: R(s, a, s')
# Each inner list contains rewards for reaching [s0, s1, s2].

rewards = [
    # State s0
    [
        [+10, 0, 0],  # Action a0: reward +10 when returning to s0
        [0, 0, 0],    # Action a1: no reward
        [0, 0, 0],    # Action a2: no reward
    ],

    # State s1
    [
        [0, 0, 0],     # Action a0: no reward
        [0, 0, 0],     # Placeholder for action a1
        [0, 0, -50],   # Action a2: reward -50 when reaching s2
    ],

    # State s2
    [
        [0, 0, 0],     # Placeholder for action a0
        [+40, 0, 0],   # Action a1: reward +40 when reaching s0
        [0, 0, 0],     # Placeholder for action a2
    ]
]


# List of actions that are actually available in each state.
# s0 -> a0, a1, a2
# s1 -> a0, a2
# s2 -> a1
possible_actions = [
    [0, 1, 2],
    [0, 2],
    [1]
]


## 3. Initialize the Q-Value Table

The Q-table has:

- **Rows = states**
- **Columns = actions**

We initially set every value to negative infinity (`-∞`). This ensures impossible actions will never accidentally be selected as the best action.

Then, we set all **possible actions** to `0.0` because these are our initial Q-value estimates.


In [ ]:
# Create a 3 x 3 Q-table and initially mark every action as impossible.
Q_values = np.full((3, 3), -np.inf)

# Set the Q-values of valid actions to 0.0.
for state, actions in enumerate(possible_actions):
    Q_values[state, actions] = 0.0

# Display the initial Q-table.
Q_values


## 4. Set the Discount Factor

The discount factor is represented by:

\[
\gamma
\]

It controls how much importance the agent gives to future rewards.

Here:

\[
\gamma = 0.90
\]

So future rewards are considered important, but they are slightly discounted compared with immediate rewards.


In [ ]:
# Discount factor: controls the importance of future rewards.
gamma = 0.90


## 5. Perform Q-Value Iteration

We repeatedly update every valid Q-value using:

\[
Q_{k+1}(s,a)=\sum_{s'}T(s,a,s')
\left[R(s,a,s')+\gamma\max_{a'}Q_k(s',a')\right]
\]

### In simple words

For every state and action:

1. Consider every possible next state.
2. Get the probability of reaching that next state.
3. Get the immediate reward.
4. Find the best possible future Q-value from that next state.
5. Discount the future value using `γ`.
6. Multiply by the transition probability.
7. Add all possible outcomes together.

We repeat this process many times so the Q-values gradually converge toward the optimal values.


In [ ]:
# Repeat the Q-value update process 50 times.
for iteration in range(50):

    # Save the Q-values from the previous iteration.
    # We use these old values to calculate the new Q-values.
    Q_prev = Q_values.copy()

    # Go through each state: s0, s1, and s2.
    for s in range(3):

        # Only evaluate actions that are possible in the current state.
        for a in possible_actions[s]:

            # Calculate the new Q-value using the Q-value iteration equation.
            Q_values[s, a] = np.sum([

                # Transition probability:
                # T(s, a, s')
                transition_probabilities[s][a][sp]

                *

                # Immediate reward + discounted best future Q-value
                (
                    rewards[s][a][sp]
                    +
                    gamma * np.max(Q_prev[sp])
                )

                # Check every possible next state: s0, s1, s2.
                for sp in range(3)
            ])


## 6. Display the Final Q-Values

Each value in the table represents:

\[
Q(s,a)
\]

Meaning:

> The estimated long-term value of taking action `a` while in state `s`.

A larger Q-value means that action is better for maximizing long-term reward.


In [ ]:
# Display the Q-table after the iterations.
Q_values


## 7. Extract the Optimal Policy

Once we know the Q-values, selecting the best action is straightforward.

The optimal policy is:

\[
\pi^*(s)=\arg\max_a Q^*(s,a)
\]

`argmax` returns the **index of the largest value**.

Using `axis=1` means we find the best action separately for every state (each row of the Q-table).


In [ ]:
# Find the action with the highest Q-value in each state.
# The returned indices represent the optimal actions.
optimal_policy = Q_values.argmax(axis=1)

optimal_policy


## Final Summary

This notebook follows the complete Q-value iteration workflow:

```text
Define the MDP
      ↓
Define transition probabilities
      ↓
Define rewards
      ↓
Define possible actions
      ↓
Initialize Q-values
      ↓
Repeatedly update Q-values
      ↓
Choose the action with the highest Q-value
      ↓
Obtain the optimal policy
```

### Key idea

> **Q-value iteration learns how good each action is in each state, then chooses the action with the highest Q-value.**
